# Innovations in RL for Hard Constraint Rules
### Applied to Amazon Review Compliance

**Hard constraint** = a rule that must **never** be violated, not just "violated less often".

Soft penalties (reward shaping) are insufficient: the agent can trade off penalty against reward and *choose* to violate rules when profitable enough. Hard constraints forbid this trade-off entirely.

## Algorithms Implemented (from scratch, pure NumPy)

| # | Algorithm | Guarantee Type | Key Innovation |
|---|-----------|---------------|----------------|
| 1 | **CMDP + Lagrangian** | Approximate (converges to 0) | Auto-adjusting dual variable λ |
| 2 | **CPO-lite** | Per-update feasibility | Project gradient onto constraint manifold |
| 3 | **Shield RL** | Hard (by construction) | Formal override of violating actions |
| 4 | **Lexicographic RL** | Strict priority order | Hierarchy: safety first, reward second |
| 5 | **Feasibility-Filtered RL (RFT)** | Dataset-level | Only train on rule-compliant samples |
| 6 | **Barrier Function RL** | Repulsive boundary | Differentiable barrier warps gradient |
| 7 | **SCPO-lite** | Per-token constraint | State-wise (per-step) constraint budget |

---
**Context from previous notebook**: We have a bigram language model policy `π(next_token | prev_token)` trained to generate Amazon reviews. Rules are encoded as reward components. Here we enforce those rules as *hard constraints*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict, Counter
import re
from copy import deepcopy

np.random.seed(42)

# ─────────────────────────────────────────────
# Shared vocabulary + reward functions
# (same as rl_amazon_review_notebook)
# ─────────────────────────────────────────────

GOOD_WORDS = [
    '<BOS>', '<EOS>',
    'the', 'this', 'a', 'an', 'is', 'was', 'are', 'i', 'my', 'we',
    'product', 'item', 'quality', 'material', 'design', 'size', 'color',
    'works', 'arrived', 'fits', 'holds', 'feels', 'looks', 'lasts',
    'great', 'good', 'excellent', 'perfect', 'nice', 'solid', 'sturdy',
    'pleased', 'satisfied', 'happy', 'impressed', 'love', 'recommend',
    'fast', 'quick', 'well', 'packaged', 'delivery', 'shipping',
    'worth', 'value', 'price', 'affordable', 'durable', 'comfortable',
    'would', 'will', 'definitely', 'highly', 'really', 'very', 'so',
    'easy', 'use', 'install', 'clean', 'made', 'built',
    'expected', 'described', 'advertised', 'pictures', 'same',
]
BAD_WORDS = [
    'buy', 'click', 'discount', 'sale', 'coupon', 'promo', 'sponsored',
    'scam', 'fake', 'junk', 'garbage', 'terrible', 'hate', 'crap', 'damn',
    'free', 'complimentary', 'exchange',
]

VOCAB = GOOD_WORDS + BAD_WORDS
V = len(VOCAB)
token2id = {w: i for i, w in enumerate(VOCAB)}
id2token = {i: w for w, i in token2id.items()}
BOS_ID = token2id['<BOS>']
EOS_ID = token2id['<EOS>']
BAD_IDS = set(token2id[w] for w in BAD_WORDS)
PRODUCT_WORDS = {'product', 'item', 'quality', 'works', 'great', 'good', 'excellent',
                 'satisfied', 'happy', 'recommend', 'love', 'value', 'durable', 'comfortable'}

# ─────────────────────────────────────────────
# Reward & constraint functions
# ─────────────────────────────────────────────

def softmax(x):
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

def compute_reward(tokens):
    """Soft quality reward (higher = better review quality)."""
    words = [id2token[t] for t in tokens if t not in (BOS_ID, EOS_ID)]
    if not words: return -1.0
    topic = sum(1 for w in words if w in PRODUCT_WORDS)
    length_bonus = min(len(words) / 15.0, 1.0) * 0.5
    return min(topic * 0.12, 1.0) + length_bonus

# Hard constraint functions: return constraint violation amount ≥ 0
# Feasible means ALL constraints ≤ 0  (or == 0)

def constraint_no_bad_words(tokens) -> float:
    """C1: count of bad/promotional/profane tokens used. Must be 0."""
    return sum(1 for t in tokens if t in BAD_IDS)

def constraint_min_length(tokens, min_len=10) -> float:
    """C2: shortfall below minimum word count. Must be 0."""
    content = [t for t in tokens if t not in (BOS_ID, EOS_ID)]
    return max(0, min_len - len(content))

def constraint_on_topic(tokens, min_topic=2) -> float:
    """C3: must mention at least min_topic product-relevant words."""
    words = [id2token[t] for t in tokens if t not in (BOS_ID, EOS_ID)]
    topic_count = sum(1 for w in words if w in PRODUCT_WORDS)
    return max(0, min_topic - topic_count)

CONSTRAINTS = [
    ('No bad words',  constraint_no_bad_words,  0),   # budget d=0: zero tolerance
    ('Min length',    constraint_min_length,    0),   # budget d=0: must reach min
    ('On topic',      constraint_on_topic,      0),   # budget d=0: must mention product
]

def is_feasible(tokens) -> bool:
    return all(fn(tokens) <= d for _, fn, d in CONSTRAINTS)

def total_violation(tokens) -> float:
    return sum(max(0, fn(tokens) - d) for _, fn, d in CONSTRAINTS)

# ─────────────────────────────────────────────
# Base policy
# ─────────────────────────────────────────────

class BigramPolicy:
    def __init__(self, vocab_size=V):
        self.V = vocab_size
        self.W = np.random.randn(vocab_size, vocab_size) * 0.1
        self.W[:, EOS_ID] -= 2.0

    def log_probs(self, prev):
        return np.log(softmax(self.W[prev]) + 1e-10)

    def probs(self, prev):
        return softmax(self.W[prev])

    def generate(self, max_len=30, forbidden_ids=None):
        tokens = [BOS_ID]
        log_probs_taken = []
        for _ in range(max_len):
            p = self.probs(tokens[-1]).copy()
            if forbidden_ids:
                p[list(forbidden_ids)] = 0.0
                p /= p.sum() + 1e-10
            tok = np.random.choice(self.V, p=p)
            log_probs_taken.append(np.log(p[tok] + 1e-10))
            tokens.append(tok)
            if tok == EOS_ID: break
        return tokens, log_probs_taken

    def to_text(self, tokens):
        return ' '.join(id2token[t] for t in tokens if t not in (BOS_ID, EOS_ID))

    def clone(self):
        p = BigramPolicy(self.V)
        p.W = self.W.copy()
        return p


def score_policy(policy, n=200, max_len=30, shield_fn=None):
    rewards, violations, feasibles = [], [], []
    for _ in range(n):
        tokens, _ = policy.generate(max_len=max_len)
        if shield_fn: tokens = shield_fn(tokens, policy)
        rewards.append(compute_reward(tokens))
        violations.append(total_violation(tokens))
        feasibles.append(is_feasible(tokens))
    return {
        'reward': np.mean(rewards),
        'violation': np.mean(violations),
        'feasible_rate': np.mean(feasibles),
    }

print(f'Vocab: {V} tokens ({len(BAD_WORDS)} forbidden bad words)')
baseline_policy = BigramPolicy()
bs = score_policy(baseline_policy)
print(f'Untrained policy — reward: {bs["reward"]:.3f}, '
      f'violation: {bs["violation"]:.3f}, '
      f'feasible: {bs["feasible_rate"]:.1%}')

---
## Algorithm 1: CMDP + Lagrangian Relaxation

### Theory

A **Constrained MDP** adds a cost budget on top of the standard reward objective:

$$\max_\pi \; \mathbb{E}[R] \quad \text{subject to} \quad \mathbb{E}[C_i] \leq d_i \quad \forall i$$

The **Lagrangian** converts this to an unconstrained min-max problem:

$$\mathcal{L}(\pi, \boldsymbol{\lambda}) = \mathbb{E}[R] - \sum_i \lambda_i \big(\mathbb{E}[C_i] - d_i\big)$$

**Primal-dual training alternates:**
1. **Policy step** (max over π): `W += lr_π * ∇_π L`  → maximize reward minus penalised violations  
2. **Dual step** (max over λ): `λᵢ += lr_λ * (E[Cᵢ] - dᵢ)` → increase penalty if constraint violated

When a constraint is violated, λ rises automatically, forcing the policy to take it seriously. When satisfied, λ falls — allowing the policy to trade off reward again.

**Key advantage over fixed penalty**: No hand-tuning of weights. λ self-adjusts to the tightest value needed.

In [ ]:
class LagrangianRL:
    """
    CMDP trained with Primal-Dual (Lagrangian) policy gradient.
    
    Maintains one λ_i per constraint.
    Policy update: gradient ascent on L = R - Σ λ_i * C_i
    Dual update:   λ_i += lr_λ * (C_i - d_i)   (gradient ascent on dual)
    """
    def __init__(self, policy: BigramPolicy, lr_pi=0.02, lr_lambda=0.05,
                 lambda_init=1.0, lambda_max=50.0):
        self.policy = policy
        self.lr_pi = lr_pi
        self.lr_lambda = lr_lambda
        self.lambda_max = lambda_max
        # One multiplier per constraint
        self.lambdas = np.full(len(CONSTRAINTS), lambda_init)
        self.reward_history = []
        self.violation_history = []
        self.lambda_history = []
        self.baseline = 0.0

    def step(self, batch_size=8, max_len=28):
        grad_W = np.zeros_like(self.policy.W)
        batch_rewards, batch_violations = [], []

        episodes = []
        for _ in range(batch_size):
            tokens, _ = self.policy.generate(max_len=max_len)
            r = compute_reward(tokens)
            # Per-constraint violations
            cvs = np.array([max(0.0, fn(tokens) - d)
                            for _, fn, d in CONSTRAINTS])
            episodes.append((tokens, r, cvs))
            batch_rewards.append(r)
            batch_violations.append(cvs)

        avg_cvs = np.mean(batch_violations, axis=0)

        # Dual update: λ_i += lr_λ * (E[C_i] - d_i)
        # Since d_i = 0 for all our constraints:
        self.lambdas += self.lr_lambda * avg_cvs
        self.lambdas = np.clip(self.lambdas, 0.0, self.lambda_max)

        # Policy update using Lagrangian objective as signal
        self.baseline = 0.95 * self.baseline + 0.05 * np.mean(batch_rewards)

        for tokens, r, cvs in episodes:
            # Lagrangian signal: reward minus penalised constraint violations
            lagrangian_signal = r - np.dot(self.lambdas, cvs)
            advantage = lagrangian_signal - self.baseline

            for t in range(len(tokens) - 1):
                prev, nxt = tokens[t], tokens[t + 1]
                p = self.policy.probs(prev)
                oh = np.zeros(self.policy.V); oh[nxt] = 1.0
                grad_W[prev] += advantage * (oh - p)

        self.policy.W += self.lr_pi * grad_W / batch_size

        avg_r = np.mean(batch_rewards)
        avg_v = np.mean([total_violation(ep[0]) for ep in episodes])
        self.reward_history.append(avg_r)
        self.violation_history.append(avg_v)
        self.lambda_history.append(self.lambdas.copy())

        return {'reward': avg_r, 'violation': avg_v, 'lambdas': self.lambdas.copy()}


N_STEPS = 300
print('Training Lagrangian RL...')
lagrangian = LagrangianRL(BigramPolicy(), lr_pi=0.02, lr_lambda=0.1)
for _ in range(N_STEPS):
    lagrangian.step()

s = score_policy(lagrangian.policy)
print(f'Lagrangian RL — reward: {s["reward"]:.3f}, '
      f'violation: {s["violation"]:.3f}, '
      f'feasible: {s["feasible_rate"]:.1%}')
print(f'Final λ values: {[f"{n}={v:.2f}" for (n,_,__), v in zip(CONSTRAINTS, lagrangian.lambdas)]}')

---
## Algorithm 2: CPO-lite — Constrained Policy Optimization

### Theory

CPO (Achiam et al. 2017) takes a trust-region update but **projects** the gradient onto the constraint-feasible manifold before applying it.

The core idea: compute the unconstrained policy gradient $g$ and constraint gradient $c$, then find the component of $g$ that is **orthogonal to $c$** — i.e., the part of the reward gradient that doesn't push the policy toward violations:

$$g_\text{safe} = g - \frac{g^\top c}{\|c\|^2} c \quad \text{(project out the unsafe direction)}$$

If the constraint is already violated, add a correction step that moves *toward* the feasible region:

$$\Delta\theta = g_\text{safe} + \alpha \cdot (-c) \quad \text{if } \mathbb{E}[C] > d$$

In [ ]:
class CPOLite:
    """
    Constrained Policy Optimization (simplified).
    Projects the reward gradient onto the null-space of constraint gradients.
    If constraint is violated, adds a feasibility recovery step.
    """
    def __init__(self, policy: BigramPolicy, lr=0.02, correction_coef=2.0,
                 batch_size=8):
        self.policy = policy
        self.lr = lr
        self.correction_coef = correction_coef
        self.batch_size = batch_size
        self.reward_history = []
        self.violation_history = []
        self.baseline = 0.0

    def score_grad(self, tokens, signal):
        """Compute score-function gradient for a sequence given a scalar signal."""
        g = np.zeros_like(self.policy.W)
        for t in range(len(tokens) - 1):
            prev, nxt = tokens[t], tokens[t + 1]
            p = self.policy.probs(prev)
            oh = np.zeros(self.policy.V); oh[nxt] = 1.0
            g[prev] += signal * (oh - p)
        return g

    def step(self, max_len=28):
        rewards, violations, eps = [], [], []
        for _ in range(self.batch_size):
            tokens, _ = self.policy.generate(max_len=max_len)
            r = compute_reward(tokens)
            v = total_violation(tokens)
            rewards.append(r); violations.append(v); eps.append(tokens)

        avg_r = np.mean(rewards)
        avg_v = np.mean(violations)
        self.baseline = 0.95 * self.baseline + 0.05 * avg_r

        # Reward gradient (unconstrained)
        g_reward = np.zeros_like(self.policy.W)
        for tokens, r in zip(eps, rewards):
            g_reward += self.score_grad(tokens, r - self.baseline)
        g_reward /= self.batch_size

        # Constraint gradient (direction of increasing violation)
        g_constraint = np.zeros_like(self.policy.W)
        for tokens, v in zip(eps, violations):
            g_constraint += self.score_grad(tokens, v)
        g_constraint /= self.batch_size

        c_norm_sq = np.sum(g_constraint ** 2) + 1e-10

        # Project reward gradient onto null-space of constraint gradient
        # g_safe = g_reward - (g_reward · g_constraint / ||g_constraint||²) * g_constraint
        dot = np.sum(g_reward * g_constraint)
        g_safe = g_reward - (dot / c_norm_sq) * g_constraint

        # If constraint violated: add recovery step toward feasibility
        if avg_v > 0:
            # Move against the constraint gradient (reduce violations)
            g_safe -= self.correction_coef * g_constraint

        self.policy.W += self.lr * g_safe
        self.reward_history.append(avg_r)
        self.violation_history.append(avg_v)

        return {'reward': avg_r, 'violation': avg_v}


print('Training CPO-lite...')
cpo = CPOLite(BigramPolicy(), lr=0.02, correction_coef=2.5, batch_size=8)
for _ in range(N_STEPS):
    cpo.step()

s = score_policy(cpo.policy)
print(f'CPO-lite — reward: {s["reward"]:.3f}, '
      f'violation: {s["violation"]:.3f}, '
      f'feasible: {s["feasible_rate"]:.1%}')

---
## Algorithm 3: Shield RL

### Theory

A **shield** provides a formal hard guarantee *regardless of the policy's internal state*. It sits between the policy output and the environment, intercepting any action that would violate a rule and replacing it with the nearest safe action.

```
Policy π(a|s) ──┐
                 ▼
             [SHIELD]  ── if action violates rules → override with safe action
                 │
                 ▼
           Environment
```

**Properties:**
- **Hard guarantee** by construction — no learning required
- Can be applied to *any* existing policy with zero retraining
- The policy *inside* still learns, but its outputs are always filtered

**Types:**
- **Restrictive shield**: Replace violating action with most-probable safe action
- **Permissive shield**: Allow any safe action (maximally non-interfering)
- **Minimal-intervention shield**: Choose the safe action closest to the policy's intended action

In [ ]:
class Shield:
    """
    Hard safety shield for the bigram token policy.
    Intercepts token generation and overrides unsafe choices.
    Three modes: restrictive, permissive, minimal-intervention.
    """
    def __init__(self, mode='minimal'):
        assert mode in ('restrictive', 'permissive', 'minimal')
        self.mode = mode
        self.interventions = 0
        self.total_tokens = 0

    def is_safe_token(self, token_id, tokens_so_far):
        """Check if emitting this token keeps the sequence on a safe path."""
        return token_id not in BAD_IDS

    def safe_set(self, tokens_so_far, policy_probs):
        """Return set of tokens that are safe to emit given context."""
        return [t for t in range(V) if self.is_safe_token(t, tokens_so_far)]

    def intervene(self, intended_token, tokens_so_far, policy_probs):
        """
        Given the policy's intended token, return the shielded token.
        Returns (shielded_token, was_overridden).
        """
        if self.is_safe_token(intended_token, tokens_so_far):
            return intended_token, False

        safe = self.safe_set(tokens_so_far, policy_probs)
        if not safe:
            return EOS_ID, True  # Force end if no safe tokens exist

        if self.mode == 'restrictive':
            # Always use the highest-probability safe token
            safe_probs = {t: policy_probs[t] for t in safe}
            return max(safe_probs, key=safe_probs.get), True

        elif self.mode == 'permissive':
            # Resample uniformly from all safe tokens
            return np.random.choice(safe), True

        else:  # minimal intervention
            # Use normalised policy distribution over safe tokens only
            safe_arr = np.array(safe)
            safe_p = policy_probs[safe_arr]
            safe_p = safe_p / safe_p.sum()
            return int(np.random.choice(safe_arr, p=safe_p)), True

    def shielded_generate(self, policy, max_len=30):
        """Generate with hard shield enforcement."""
        tokens = [BOS_ID]
        for _ in range(max_len):
            p = policy.probs(tokens[-1])
            intended = np.random.choice(V, p=p)
            shielded, overridden = self.intervene(intended, tokens, p)
            self.total_tokens += 1
            if overridden:
                self.interventions += 1
            tokens.append(shielded)
            if shielded == EOS_ID:
                break
        return tokens

    @property
    def intervention_rate(self):
        return self.interventions / max(1, self.total_tokens)


# Demonstrate: same untrained policy, three shield modes
untrained = BigramPolicy()

print('Shield modes applied to an untrained policy:')
print(f'{"Mode":<22} {"Reward":>8} {"Violation":>10} {"Feasible%":>10} {"Overrides%":>12}')
print('-' * 65)

for mode in ('restrictive', 'permissive', 'minimal'):
    shield = Shield(mode=mode)
    rewards, feasibles, violations = [], [], []
    for _ in range(300):
        tokens = shield.shielded_generate(untrained, max_len=28)
        rewards.append(compute_reward(tokens))
        feasibles.append(is_feasible(tokens))
        violations.append(total_violation(tokens))
    print(f'{mode:<22} {np.mean(rewards):>8.3f} {np.mean(violations):>10.3f} '
          f'{np.mean(feasibles):>10.1%} {shield.intervention_rate:>12.1%}')

print()
print('Note: constraint_no_bad_words is 100% satisfied by shield (hard guarantee).')
print('Remaining violations come from length/topic constraints — not covered by this shield.')

In [ ]:
# Shield + RL together: shield handles hard safety, RL optimises quality
class ShieldedRL:
    """
    Combine Shield (hard guarantee on token safety) with
    REINFORCE (soft reward optimisation for quality).
    The policy never generates bad tokens; RL pushes it toward longer,
    more on-topic reviews.
    """
    def __init__(self, policy, shield, lr=0.02):
        self.policy = policy
        self.shield = shield
        self.lr = lr
        self.baseline = 0.0
        self.reward_history = []
        self.violation_history = []

    def step(self, batch_size=8, max_len=28):
        grad_W = np.zeros_like(self.policy.W)
        rewards, violations = [], []

        for _ in range(batch_size):
            tokens = self.shield.shielded_generate(self.policy, max_len=max_len)
            r = compute_reward(tokens)
            v = total_violation(tokens)
            rewards.append(r); violations.append(v)
            adv = r - self.baseline
            for t in range(len(tokens) - 1):
                prev, nxt = tokens[t], tokens[t + 1]
                p = self.policy.probs(prev)
                oh = np.zeros(self.policy.V); oh[nxt] = 1.0
                grad_W[prev] += adv * (oh - p)

        self.policy.W += self.lr * grad_W / batch_size
        avg_r = np.mean(rewards)
        self.baseline = 0.95 * self.baseline + 0.05 * avg_r
        self.reward_history.append(avg_r)
        self.violation_history.append(np.mean(violations))
        return {'reward': avg_r, 'violation': np.mean(violations)}


print('Training Shield + RL (shield handles bad words, RL improves quality)...')
shield_rl = ShieldedRL(BigramPolicy(), Shield(mode='minimal'), lr=0.02)
for _ in range(N_STEPS):
    shield_rl.step()

s = score_policy(shield_rl.policy,
    shield_fn=lambda t, p: Shield(mode='minimal').shielded_generate(p, max_len=30))
print(f'Shield+RL — reward: {s["reward"]:.3f}, '
      f'violation: {s["violation"]:.3f}, '
      f'feasible: {s["feasible_rate"]:.1%}')

---
## Algorithm 4: Lexicographic Policy Optimization

### Theory

In standard RL, all objectives are combined into one scalar. In **Lexicographic RL**, objectives have a strict *priority order* — like a dictionary where you compare first letters before second letters.

**Training stages:**
1. **Stage 1**: Optimize constraint satisfaction only. Train until convergence.
2. **Stage 2**: Optimize reward quality, *subject to* not reducing constraint satisfaction below a threshold ε.

The key mechanism is **gradient filtering**: in Stage 2, any gradient component that would harm Stage 1's objective is projected out before applying the update.

$$g^{(2)}_\text{filtered} = g^{(2)} - \frac{\langle g^{(2)}, g^{(1)} \rangle}{\|g^{(1)}\|^2 + \epsilon} \cdot g^{(1)}$$

This is equivalent to Pareto dominance: never trade off a higher-priority objective for a lower-priority one.

In [ ]:
class LexicographicRL:
    """
    Two-stage lexicographic RL:
      Stage 1: Optimise constraint satisfaction (feasibility rate)
      Stage 2: Optimise reward quality, with gradient filtered
               to not degrade constraint satisfaction.
    """
    def __init__(self, policy, lr=0.02, stage1_steps=150, eps=0.05):
        self.policy = policy
        self.lr = lr
        self.stage1_steps = stage1_steps
        self.eps = eps
        self.stage = 1
        self.step_count = 0
        self.reward_history = []
        self.violation_history = []
        self.baseline_r = 0.0
        self.baseline_v = 0.0

    def score_grad(self, episodes, signal_fn):
        g = np.zeros_like(self.policy.W)
        for tokens in episodes:
            sig = signal_fn(tokens)
            for t in range(len(tokens) - 1):
                prev, nxt = tokens[t], tokens[t + 1]
                p = self.policy.probs(prev)
                oh = np.zeros(self.policy.V); oh[nxt] = 1.0
                g[prev] += sig * (oh - p)
        return g / len(episodes)

    def step(self, batch_size=8, max_len=28):
        if self.step_count == self.stage1_steps:
            self.stage = 2
            print(f'  → Switching to Stage 2 (reward optimisation with gradient filtering)')

        episodes = [self.policy.generate(max_len=max_len)[0]
                    for _ in range(batch_size)]
        rewards = [compute_reward(t) for t in episodes]
        violations = [total_violation(t) for t in episodes]

        avg_r = np.mean(rewards)
        avg_v = np.mean(violations)
        self.baseline_r = 0.95 * self.baseline_r + 0.05 * avg_r
        self.baseline_v = 0.95 * self.baseline_v + 0.05 * avg_v

        if self.stage == 1:
            # Stage 1: only reduce violations (negative signal on violations)
            g = self.score_grad(episodes,
                                lambda t: -(total_violation(t) - self.baseline_v))
            self.policy.W += self.lr * g

        else:  # stage == 2
            # Compute reward gradient and constraint gradient
            g_reward = self.score_grad(episodes,
                                       lambda t: compute_reward(t) - self.baseline_r)
            g_constraint = self.score_grad(episodes,
                                           lambda t: -(total_violation(t) - self.baseline_v))

            # Filter g_reward: remove component along g_constraint
            # This ensures reward optimisation doesn't undo safety gains
            c_norm_sq = np.sum(g_constraint ** 2) + 1e-10
            dot = np.sum(g_reward * g_constraint)
            g_filtered = g_reward - (dot / c_norm_sq) * g_constraint

            # Apply filtered gradient + small safety gradient
            self.policy.W += self.lr * (g_filtered + self.eps * g_constraint)

        self.reward_history.append(avg_r)
        self.violation_history.append(avg_v)
        self.step_count += 1


print('Training Lexicographic RL (Stage 1: safety, Stage 2: quality)...')
lexicographic = LexicographicRL(BigramPolicy(), lr=0.025, stage1_steps=150)
for _ in range(N_STEPS):
    lexicographic.step()

s = score_policy(lexicographic.policy)
print(f'Lexicographic RL — reward: {s["reward"]:.3f}, '
      f'violation: {s["violation"]:.3f}, '
      f'feasible: {s["feasible_rate"]:.1%}')

---
## Algorithm 5: Feasibility-Filtered RL (RFT / BOND-style)

### Theory

**Rejection Sampling Fine-Tuning (RFT)** is conceptually the simplest hard-constraint approach:

1. Sample a large batch of candidates from the current policy  
2. **Reject all that violate any hard constraint** — throw them away entirely  
3. Fine-tune (supervised) on the accepted, feasible candidates  
4. Repeat  

This is exactly what **DeepSeek-R1** does with math problems: generate 8–16 reasoning chains, keep only those that produce the correct verified answer, train on those.

**BOND (Best-of-N Distillation)** is a variant: sample N, rank by reward, distil the *distribution* of the top-K into the policy.

**Why it works**: By construction, the training dataset contains *only* feasible examples. The policy cannot learn to violate rules because it never sees rule-violating gradients.

In [ ]:
class FeasibilityFilteredRL:
    """
    RFT-style training:
      1. Sample large candidate pool
      2. Keep only feasible candidates
      3. Among feasible candidates, select top-K by reward
      4. Behaviour cloning gradient on top-K
    
    If no feasible candidates: skip update (safety preserved).
    """
    def __init__(self, policy, lr=0.02, pool_size=32, top_k=8):
        self.policy = policy
        self.lr = lr
        self.pool_size = pool_size
        self.top_k = top_k
        self.reward_history = []
        self.violation_history = []
        self.acceptance_rate_history = []

    def step(self, max_len=28):
        # Step 1: Sample large pool
        pool = []
        for _ in range(self.pool_size):
            tokens, _ = self.policy.generate(max_len=max_len)
            r = compute_reward(tokens)
            v = total_violation(tokens)
            feasible = is_feasible(tokens)
            pool.append({'tokens': tokens, 'reward': r, 'violation': v, 'feasible': feasible})

        # Step 2: Filter — keep only feasible
        feasible_pool = [ep for ep in pool if ep['feasible']]
        acceptance_rate = len(feasible_pool) / self.pool_size

        avg_v = np.mean([ep['violation'] for ep in pool])
        avg_r = np.mean([ep['reward'] for ep in pool])

        if not feasible_pool:
            # No feasible samples — no update (preserve safety, avoid bad gradients)
            self.reward_history.append(avg_r)
            self.violation_history.append(avg_v)
            self.acceptance_rate_history.append(0.0)
            return {'reward': avg_r, 'violation': avg_v, 'acceptance': 0.0}

        # Step 3: Rank feasible by reward, take top-K
        feasible_pool.sort(key=lambda ep: ep['reward'], reverse=True)
        top_k = feasible_pool[:self.top_k]

        # Step 4: Behaviour cloning on top-K (maximise log-prob of their tokens)
        grad_W = np.zeros_like(self.policy.W)
        for ep in top_k:
            tokens = ep['tokens']
            # Weight by reward (better reviews get stronger signal)
            weight = ep['reward']
            for t in range(len(tokens) - 1):
                prev, nxt = tokens[t], tokens[t + 1]
                p = self.policy.probs(prev)
                oh = np.zeros(self.policy.V); oh[nxt] = 1.0
                grad_W[prev] += weight * (oh - p)

        self.policy.W += self.lr * grad_W / len(top_k)

        self.reward_history.append(avg_r)
        self.violation_history.append(avg_v)
        self.acceptance_rate_history.append(acceptance_rate)
        return {'reward': avg_r, 'violation': avg_v, 'acceptance': acceptance_rate}


print('Training Feasibility-Filtered RL (RFT style)...')
rft = FeasibilityFilteredRL(BigramPolicy(), lr=0.02, pool_size=40, top_k=8)
for _ in range(N_STEPS):
    rft.step()

s = score_policy(rft.policy)
print(f'RFT — reward: {s["reward"]:.3f}, '
      f'violation: {s["violation"]:.3f}, '
      f'feasible: {s["feasible_rate"]:.1%}')
print(f'Final acceptance rate: {np.mean(rft.acceptance_rate_history[-50:]):.1%}')

---
## Algorithm 6: Barrier Function RL

### Theory

**Control Barrier Functions** (CBFs) define a "safety zone" $\mathcal{C} = \{x : h(x) \geq 0\}$. The key insight is that instead of penalising constraint violations *after* they happen, a barrier function **repels** the policy *before* it crosses the boundary.

The barrier penalty becomes infinite as you approach violation:

$$B(x) = -\log(h(x)) \quad \text{(log-barrier)}$$

$$B(x) = \frac{1}{h(x)} \quad \text{(reciprocal barrier)}$$

This is applied as an additive term in the RL objective:

$$\mathcal{L}(\pi) = \mathbb{E}[R] - \beta \cdot B(\mathbb{E}[C])$$

Unlike fixed penalty coefficients, the barrier gradient automatically scales to infinity near the constraint boundary — providing an increasingly strong "wall".

In [ ]:
class BarrierFunctionRL:
    """
    RL with log-barrier on constraint violations.
    Barrier B(v) = -log(max(budget - violation, eps))
    Gradient becomes very large as violation → budget.
    """
    def __init__(self, policy, lr=0.02, beta=0.5, batch_size=8):
        self.policy = policy
        self.lr = lr
        self.beta = beta
        self.batch_size = batch_size
        self.baseline_r = 0.0
        self.reward_history = []
        self.violation_history = []

    def barrier(self, violation, budget=0.0, margin=0.1):
        """Log-barrier: -log(budget + margin - violation). Increases steeply near budget."""
        slack = budget + margin - violation
        slack = max(slack, 1e-4)
        return -np.log(slack)

    def barrier_grad(self, violation, budget=0.0, margin=0.1):
        """Gradient of barrier w.r.t. violation: 1 / (budget + margin - violation)."""
        slack = budget + margin - violation
        slack = max(slack, 1e-4)
        return 1.0 / slack  # Positive: increasing violation increases barrier

    def step(self, max_len=28):
        grad_W = np.zeros_like(self.policy.W)
        rewards, violations = [], []

        for _ in range(self.batch_size):
            tokens, _ = self.policy.generate(max_len=max_len)
            r = compute_reward(tokens)
            v = total_violation(tokens)
            rewards.append(r); violations.append(v)

            # Combined signal: reward minus beta * barrier gradient * violation
            # The barrier gradient increases as violation approaches budget
            bg = self.barrier_grad(v)
            signal = (r - self.baseline_r) - self.beta * bg * v

            for t in range(len(tokens) - 1):
                prev, nxt = tokens[t], tokens[t + 1]
                p = self.policy.probs(prev)
                oh = np.zeros(self.policy.V); oh[nxt] = 1.0
                grad_W[prev] += signal * (oh - p)

        self.policy.W += self.lr * grad_W / self.batch_size

        avg_r = np.mean(rewards)
        avg_v = np.mean(violations)
        self.baseline_r = 0.95 * self.baseline_r + 0.05 * avg_r
        self.reward_history.append(avg_r)
        self.violation_history.append(avg_v)


print('Training Barrier Function RL...')
barrier_rl = BarrierFunctionRL(BigramPolicy(), lr=0.02, beta=0.8, batch_size=8)
for _ in range(N_STEPS):
    barrier_rl.step()

s = score_policy(barrier_rl.policy)
print(f'Barrier RL — reward: {s["reward"]:.3f}, '
      f'violation: {s["violation"]:.3f}, '
      f'feasible: {s["feasible_rate"]:.1%}')

---
## Algorithm 7: SCPO-lite — State-Wise Constrained Policy Optimization

### Theory

Standard CMDP enforces constraints *on average* across trajectories: $\mathbb{E}[\sum_t C_t] \leq d$. This means a single step could violate a rule as long as other steps compensate.

**SCPO** enforces constraints at *every state*:

$$\mathbb{E}[C(s_t, a_t) \mid s_t = s] \leq d(s) \quad \forall s$$

**Adaptation to token generation**: At each position $t$ in the sequence, we maintain a running constraint budget. If a token would cause the per-step constraint to exceed the local budget, the gradient is blocked for that token transition.

This provides **per-position** guarantees — much stronger than trajectory-level averaging.

In [ ]:
class SCPOLite:
    """
    State-Wise Constrained Policy Optimization (simplified).
    
    Per-token budget: at each position t, the expected constraint
    cost of emitting token a must be ≤ local_budget.
    Gradient updates are gated: only propagate through token transitions
    that are locally safe.
    """
    def __init__(self, policy, lr=0.02, local_budget=0.2, batch_size=8):
        self.policy = policy
        self.lr = lr
        self.local_budget = local_budget  # Per-token violation budget
        self.batch_size = batch_size
        self.baseline = 0.0
        self.reward_history = []
        self.violation_history = []
        # Per-(prev_token, next_token) constraint cost estimate
        self.cost_estimates = np.zeros((V, V))
        self.cost_counts = np.ones((V, V)) * 10  # Pseudocount init

    def local_violation_cost(self, prev_tok, next_tok):
        """Estimated per-step violation cost of transition prev→next."""
        return next_tok in BAD_IDS  # 1.0 if bad token, 0.0 otherwise

    def update_cost_estimates(self, tokens):
        """Update running estimates of per-transition cost."""
        for t in range(len(tokens) - 1):
            prev, nxt = tokens[t], tokens[t + 1]
            cost = float(nxt in BAD_IDS)
            n = self.cost_counts[prev, nxt]
            self.cost_estimates[prev, nxt] = (self.cost_estimates[prev, nxt] * n + cost) / (n + 1)
            self.cost_counts[prev, nxt] += 1

    def is_locally_safe(self, prev_tok, next_tok):
        """Check whether this transition is within per-state budget."""
        return self.cost_estimates[prev_tok, next_tok] <= self.local_budget

    def step(self, max_len=28):
        grad_W = np.zeros_like(self.policy.W)
        rewards, violations = [], []

        for _ in range(self.batch_size):
            tokens, _ = self.policy.generate(max_len=max_len)
            r = compute_reward(tokens)
            v = total_violation(tokens)
            rewards.append(r); violations.append(v)

            self.update_cost_estimates(tokens)

            adv = r - self.baseline
            for t in range(len(tokens) - 1):
                prev, nxt = tokens[t], tokens[t + 1]

                # State-wise gate: only update this transition if it's locally safe
                # OR if we're penalising it (negative advantage for violating transitions)
                locally_safe = self.is_locally_safe(prev, nxt)
                is_violation = nxt in BAD_IDS

                if locally_safe and not is_violation:
                    # Safe transition: apply normal reward gradient
                    p = self.policy.probs(prev)
                    oh = np.zeros(self.policy.V); oh[nxt] = 1.0
                    grad_W[prev] += adv * (oh - p)
                elif is_violation:
                    # Violation: always penalise regardless of reward
                    p = self.policy.probs(prev)
                    oh = np.zeros(self.policy.V); oh[nxt] = 1.0
                    grad_W[prev] -= 2.0 * (oh - p)  # Push away from this transition

        self.policy.W += self.lr * grad_W / self.batch_size

        avg_r = np.mean(rewards)
        avg_v = np.mean(violations)
        self.baseline = 0.95 * self.baseline + 0.05 * avg_r
        self.reward_history.append(avg_r)
        self.violation_history.append(avg_v)


print('Training SCPO-lite...')
scpo = SCPOLite(BigramPolicy(), lr=0.02, local_budget=0.1, batch_size=8)
for _ in range(N_STEPS):
    scpo.step()

s = score_policy(scpo.policy)
print(f'SCPO-lite — reward: {s["reward"]:.3f}, '
      f'violation: {s["violation"]:.3f}, '
      f'feasible: {s["feasible_rate"]:.1%}')

---
## Comparison: All Algorithms

In [ ]:
# ─────────────────────────────────────────────
# Violation curves over training
# ─────────────────────────────────────────────

def smooth(arr, k=20):
    return np.convolve(arr, np.ones(k) / k, mode='valid')

algos = [
    ('Lagrangian',   lagrangian,   '#e74c3c'),
    ('CPO-lite',     cpo,          '#3498db'),
    ('Shield+RL',    shield_rl,    '#2ecc71'),
    ('Lexicographic',lexicographic,'#9b59b6'),
    ('RFT',          rft,          '#f39c12'),
    ('Barrier',      barrier_rl,   '#1abc9c'),
    ('SCPO-lite',    scpo,         '#e67e22'),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
for name, algo, col in algos:
    v = smooth(algo.violation_history[:N_STEPS])
    ax.plot(v, label=name, color=col, linewidth=1.8)
ax.set_xlabel('Training Step')
ax.set_ylabel('Average Constraint Violation')
ax.set_title('Constraint Violation Over Training')
ax.axhline(0, color='black', linestyle='--', linewidth=1.0, alpha=0.6, label='Hard limit (0)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax.set_ylim(-0.1, None)

ax2 = axes[1]
for name, algo, col in algos:
    r = smooth(algo.reward_history[:N_STEPS])
    ax2.plot(r, label=name, color=col, linewidth=1.8)
ax2.set_xlabel('Training Step')
ax2.set_ylabel('Average Reward')
ax2.set_title('Reward Quality Over Training')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('hard_constraint_training.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# Final evaluation: reward vs feasibility scatter
# ─────────────────────────────────────────────

print(f'\n{"Algorithm":<18} {"Reward":>8} {"Violation":>10} {"Feasible%":>10} {"Guarantee":>12}')
print('-' * 62)

GUARANTEES = {
    'Random':        'None',
    'Lagrangian':    'Approximate',
    'CPO-lite':      'Per-update',
    'Shield+RL':     'Hard (token)',
    'Lexicographic': 'Priority',
    'RFT':           'Dataset-level',
    'Barrier':       'Repulsive',
    'SCPO-lite':     'Per-state',
}

results = {}
for name, algo, _ in [('Random', type('', (), {'policy': BigramPolicy()})(), '')]  + \
                      [(n, a, c) for n, a, c in algos]:
    if name == 'Shield+RL':
        shield = Shield(mode='minimal')
        eval_fn = lambda t, p: shield.shielded_generate(p, max_len=28)
        s = score_policy(algo.policy, shield_fn=eval_fn)
    else:
        s = score_policy(algo.policy if hasattr(algo, 'policy') else algo)
    results[name] = s
    g = GUARANTEES.get(name, '-')
    print(f'{name:<18} {s["reward"]:>8.3f} {s["violation"]:>10.3f} '
          f'{s["feasible_rate"]:>10.1%} {g:>12}')

# Scatter: reward vs feasibility
fig, ax = plt.subplots(figsize=(9, 6))
colors_map = dict([(n, c) for n, _, c in algos] + [('Random', '#95a5a6')])

for name, s in results.items():
    col = colors_map.get(name, '#333')
    ax.scatter(s['feasible_rate'], s['reward'], s=180, color=col,
               zorder=5, edgecolors='black', linewidths=0.8)
    ax.annotate(name, (s['feasible_rate'], s['reward']),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.set_xlabel('Feasibility Rate (fraction of reviews satisfying ALL hard constraints)')
ax.set_ylabel('Avg Reward Quality')
ax.set_title('Reward vs Hard Constraint Satisfaction\n(top-right corner = best)')
ax.axvline(0.9, color='red', linestyle='--', alpha=0.4, label='90% feasibility threshold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('hard_constraint_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Algorithm Design Guide

### How to choose the right hard-constraint algorithm

```
                    ┌──────────────────────────────────────────────────────┐
                    │  Do you need a formal hard guarantee (zero violation)?│
                    └──────────────────────────────────────────────────────┘
                              │ YES                        │ NO
                              ▼                            ▼
               ┌──────────────────────┐       ┌──────────────────────────┐
               │ Can you enumerate    │       │ Do rules have a natural   │
               │ all safe actions at  │       │ priority/hierarchy?       │
               │ each step?           │       └──────────────────────────┘
               └──────────────────────┘          │ YES          │ NO
                    │ YES    │ NO                 ▼              ▼
                    ▼        ▼            Lexicographic     Lagrangian RLHF
                 Shield     CBF              RL             (auto-λ)
                 Synthesis  (continuous                        │
                            control)                          ▼
                                                    Is sample efficiency
                                                    critical?
                                                    │ NO           │ YES
                                                    ▼              ▼
                                                  RFT/BOND      CPO / SCPO
                                                 (filter)      (projection)
```

### Property Comparison

| Algorithm | Hard Guarantee | Sample Efficiency | Implementation | Best For |
|-----------|:-:|:-:|:-:|---|
| Lagrangian RLHF | ✗ (approx) | ●●● | Easy | Multi-rule LLM alignment |
| CPO-lite | ✓ (per update) | ●●○ | Medium | Multi-constraint tabular/discrete |
| **Shield RL** | **✓ (absolute)** | ●●● | Easy | Safety-critical, enumerable actions |
| Lexicographic RL | ✓ (priority 1) | ●●○ | Medium | Hierarchical rule systems |
| **RFT / BOND** | **✓ (dataset)** | ●○○ | Easy | DeepSeek-R1, code, math verification |
| Barrier Function | ✗ (soft boundary) | ●●● | Easy | Smooth constraint approaching |
| SCPO-lite | ✓ (per-state) | ●○○ | Hard | State-granular safety |

### Production LLM Recipe (combining approaches)

```python
# Step 1: SFT on human-written compliant reviews (cold start)
model = supervised_finetune(model, compliant_review_dataset)

# Step 2: RFT — generate N reviews per product, keep feasible ones
for round in range(3):
    candidates = [model.generate(product) for _ in range(16)]
    feasible   = [r for r in candidates if passes_all_amazon_rules(r)]
    model      = finetune(model, top_k_by_reward(feasible, k=4))

# Step 3: Lagrangian GRPO — online RL with auto-adjusting rule penalties
lagrangian_grpo(model, rule_reward_fns, group_size=8, kl_ref=sft_model)

# Step 4: Shield at inference — hard constrained decoding
output = shield_decode(model, prompt, forbidden_tokens=RULE_VIOLATION_TOKENS)
```

In [ ]:
# ─────────────────────────────────────────────
# Summary visualization: Algorithm properties
# ─────────────────────────────────────────────

PROPERTIES = ['Hard Guarantee', 'Sample Efficiency', 'Scalability',
              'Interpretability', 'Ease of Implementation']

SCORES = {
    'Lagrangian':    [2, 4, 5, 3, 5],
    'CPO-lite':      [4, 3, 3, 3, 3],
    'Shield RL':     [5, 4, 4, 5, 4],
    'Lexicographic': [4, 3, 3, 5, 3],
    'RFT/BOND':      [5, 2, 4, 5, 5],
    'Barrier Fn':    [3, 4, 4, 4, 4],
    'SCPO-lite':     [5, 2, 2, 4, 2],
}

N = len(PROPERTIES)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

colors_list = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12', '#1abc9c', '#e67e22']
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'polar': True})

for (name, scores), col in zip(SCORES.items(), colors_list):
    values = scores + scores[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=name, color=col)
    ax.fill(angles, values, alpha=0.06, color=col)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(PROPERTIES, size=10)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1', '2', '3', '4', '5'], size=7)
ax.set_title('Hard Constraint Algorithms:\nProperty Radar Chart', pad=20, size=13)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hard_constraint_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figures saved: hard_constraint_training.png, hard_constraint_scatter.png, hard_constraint_radar.png')